# COG Conversion with Metadata (Python API)

Convert GeoTIFF files to Cloud Optimized GeoTIFF (COG) with embedded metadata tags.

**Features:**
- Auto-detect activation event metadata from filenames **or** assign manually
- Preserves original compression settings from source files
- Validates COG structure after creation
- S3-to-S3 workflow via boto3

**Workflow:**
1. Configure S3 paths and metadata
2. Connect to AWS and list source files
3. Define metadata (auto or manual)
4. Convert to COG with metadata
5. Validate and upload
6. Verify results

## Step 1: Configuration

In [ ]:
# ========================================
# S3 Configuration
# ========================================
SOURCE_BUCKET = "nasa-disasters"
SOURCE_PREFIX = "drcs_activations_new/YOUR_PRODUCT/path"

DESTINATION_BUCKET = "nasa-disasters-staging"
DESTINATION_PREFIX = "ProgramData/YOUR_PRODUCT/Output/"  # Must end with /

AWS_REGION = "us-west-2"

# ========================================
# Metadata Mode: "auto" or "manual"
# ========================================
METADATA_MODE = "manual"  # "auto" to detect from filename, "manual" to use values below

# Manual metadata (used when METADATA_MODE = "manual")
MANUAL_METADATA = {
    "ACTIVATION_EVENT": "202501_Flood_CA",
    "YEAR_MONTH": "202501",
    "HAZARD": "Flood",
    "LOCATION": "CA",
    "PROCESSOR": "NASA Disasters COG Processor v1.0",
    # Add any custom key-value pairs here
}

# Auto-detection pattern (used when METADATA_MODE = "auto")
# Expects filenames like: YYYYMM_HAZARD_LOCATION_*.tif
AUTO_DETECT_PATTERN = r'^(\d{6})_([A-Za-z0-9]+)_([A-Za-z0-9]+)_(.*)\.tif$'

# ========================================
# Processing Options
# ========================================
OVERWRITE = False
DELETE_AFTER_MOVE = False

print(f"Mode: {METADATA_MODE}")
print(f"Source: s3://{SOURCE_BUCKET}/{SOURCE_PREFIX}")
print(f"Destination: s3://{DESTINATION_BUCKET}/{DESTINATION_PREFIX}")

## Step 2: Setup

Import libraries and connect to AWS.

In [ ]:
import sys, os, re, time
from datetime import datetime

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import boto3
from shared_utils.cog_metadata import (
    create_cog_with_metadata,
    read_compression_settings,
    validate_cog_in_memory,
)

# AWS Session
session = boto3.Session(region_name=AWS_REGION)
sts_client = session.client('sts')
identity = sts_client.get_caller_identity()

print(f"Account: {identity['Account']}")
print(f"ARN: {identity['Arn']}")

s3_client = session.client('s3')
print("\nS3 client ready")

## Step 3: List Source Files

In [ ]:
tif_files = []
paginator = s3_client.get_paginator('list_objects_v2')

for page in paginator.paginate(Bucket=SOURCE_BUCKET, Prefix=SOURCE_PREFIX):
    if 'Contents' in page:
        for obj in page['Contents']:
            if obj['Key'].endswith('.tif'):
                tif_files.append(obj['Key'])
                filename = os.path.basename(obj['Key'])
                size_gb = obj['Size'] / (1024**3)
                print(f"  {filename:<60} ({size_gb:.2f} GB)")

print(f"\nFound {len(tif_files)} .tif files")

## Step 4: Define Metadata

Choose between auto-detection from filenames or manual assignment. Edit `METADATA_MODE` in Step 1 to switch.

In [ ]:
def detect_activation_event(filename):
    """Auto-detect activation event metadata from filename pattern."""
    match = re.match(AUTO_DETECT_PATTERN, filename)
    if not match:
        return None
    year_month, hazard, location, remainder = match.groups()
    return {
        'ACTIVATION_EVENT': f"{year_month}_{hazard}_{location}",
        'YEAR_MONTH': year_month,
        'HAZARD': hazard,
        'LOCATION': location,
        'PROCESSOR': 'NASA Disasters COG Processor v1.0',
    }


def resolve_metadata(filename):
    """Return metadata dict for a file based on METADATA_MODE."""
    if METADATA_MODE == "manual":
        return dict(MANUAL_METADATA)
    else:
        detected = detect_activation_event(filename)
        if detected:
            return detected
        print(f"  WARNING: No pattern match for {filename}, using manual fallback")
        return dict(MANUAL_METADATA)


# Preview metadata for each file
print(f"Metadata mode: {METADATA_MODE}\n")
for key in tif_files:
    filename = os.path.basename(key)
    meta = resolve_metadata(filename)
    print(f"  {filename}")
    print(f"    -> {meta.get('ACTIVATION_EVENT', 'N/A')}")
    print()

## Step 5: Process Files

Downloads each file, converts to COG with metadata, validates, and uploads.

In [ ]:
results = []

for src_key in tif_files:
    filename = os.path.basename(src_key)
    dest_key = DESTINATION_PREFIX + filename
    print(f"Processing: {filename}")
    start = time.time()

    # Check if destination exists
    if not OVERWRITE:
        try:
            s3_client.head_object(Bucket=DESTINATION_BUCKET, Key=dest_key)
            print(f"  SKIPPED (exists)\n")
            results.append({'file': filename, 'status': 'skipped', 'time': 0})
            continue
        except s3_client.exceptions.ClientError:
            pass  # Does not exist, proceed

    try:
        # 1. Download to memory
        obj = s3_client.get_object(Bucket=SOURCE_BUCKET, Key=src_key)
        file_bytes = obj['Body'].read()
        print(f"  Downloaded {len(file_bytes) / (1024**2):.1f} MB")

        # 2. Resolve metadata
        metadata = resolve_metadata(filename)

        # 3. Create COG with metadata
        cog_bytes = create_cog_with_metadata(file_bytes, metadata)

        # 4. Validate
        is_valid, cog_info = validate_cog_in_memory(cog_bytes, filename)
        if is_valid:
            print(f"  COG valid: {cog_info['compression']}, {cog_info['overviews']} overviews")
        else:
            print(f"  WARNING: COG validation failed: {cog_info.get('errors', [])}")

        # 5. Upload
        s3_client.put_object(
            Bucket=DESTINATION_BUCKET, Key=dest_key,
            Body=cog_bytes, ContentType='image/tiff'
        )

        elapsed = time.time() - start
        print(f"  SUCCESS ({elapsed:.1f}s) -> s3://{DESTINATION_BUCKET}/{dest_key}\n")
        results.append({'file': filename, 'status': 'success', 'time': elapsed, 'cog_valid': is_valid})

        # 6. Optionally delete source
        if DELETE_AFTER_MOVE:
            s3_client.delete_object(Bucket=SOURCE_BUCKET, Key=src_key)
            print(f"  Deleted from source")

    except Exception as e:
        elapsed = time.time() - start
        print(f"  FAILED: {e}\n")
        results.append({'file': filename, 'status': 'failed', 'time': elapsed, 'error': str(e)})

## Step 6: Results Summary

In [ ]:
total = len(results)
success = sum(1 for r in results if r['status'] == 'success')
failed = sum(1 for r in results if r['status'] == 'failed')
skipped = sum(1 for r in results if r['status'] == 'skipped')

print(f"Total: {total}")
print(f"  Success: {success}")
print(f"  Failed:  {failed}")
print(f"  Skipped: {skipped}")

if success > 0:
    times = [r['time'] for r in results if r['status'] == 'success']
    print(f"\nAvg time: {sum(times)/len(times):.1f}s per file")

if failed > 0:
    print("\nFailed files:")
    for r in results:
        if r['status'] == 'failed':
            print(f"  - {r['file']}: {r.get('error', 'Unknown')}")

## Step 7: Verify Metadata

Download a processed file and confirm metadata tags are embedded.

In [ ]:
from rasterio.io import MemoryFile

successful = [r for r in results if r['status'] == 'success']
if successful:
    sample = successful[0]
    dest_key = DESTINATION_PREFIX + sample['file']

    print(f"Verifying: s3://{DESTINATION_BUCKET}/{dest_key}\n")

    obj = s3_client.get_object(Bucket=DESTINATION_BUCKET, Key=dest_key)
    file_bytes = obj['Body'].read()

    # Check metadata tags
    with MemoryFile(file_bytes) as memfile:
        with memfile.open() as ds:
            tags = ds.tags()

    print("Embedded metadata tags:")
    for k, v in tags.items():
        print(f"  {k}: {v}")

    # Validate COG
    is_valid, info = validate_cog_in_memory(file_bytes, sample['file'])
    print(f"\nCOG valid: {is_valid}")
    print(f"  Compression: {info.get('compression')}")
    print(f"  Tile size: {info.get('blocksize')}")
    print(f"  Overviews: {info.get('overviews')}")
else:
    print("No successful files to verify")

## Troubleshooting

- **Import errors** - Run from the `notebooks/` directory so `sys.path` finds `shared_utils`
- **No files found** - Check `SOURCE_PREFIX`, verify bucket permissions
- **Files skipped** - Set `OVERWRITE = True` in Step 1
- **COG validation warnings** - Usually harmless; check specific errors in results
- **Metadata not showing** - Use `gdalinfo your_file.tif` to inspect all metadata domains
- **Large files OOM** - For files >2GB, use the CLI version (`cog_metadata_template_cli.ipynb`) which processes on disk